# Modul 5 - Histogram, Histogram Equalization, dan Dithering
**Nama**: Mohammad Al Hafis Hidayatulloh
**NIM**: 254107023005


In [ ]:
import sys, platform, hashlib, datetime, zoneinfo

NAMA = 'Mohammad Al Hafis Hidayatulloh'
NIM = '254107023005'
N = int(NIM[-3:]) # 005 -> 5

# parameter praktikum berdasarkan akhiran NIM
PARAM = {
 'bins' : 2 ** ((N % 4) + 3), # 2 ** ((5 % 4) + 3) = 16
 'clip' : round(1.0 + (N % 5) * 0.5, 1), # 1.0 + (0) * 0.5 = 1.0
 'tile' : (N % 4) + 2, # (1) + 2 = 3
 'L'    : 2 ** ((N % 3) + 1), # 2 ** (2 + 1) = 8
}

waktu = datetime.datetime.now(zoneinfo.ZoneInfo('Asia/Jakarta'))
print('NAMA :', NAMA)
print('NIM  :', NIM, '| N =', N)
for k, v in PARAM.items():
    print(f"{k:<6} : {v}")
print('WAKTU  :', waktu.strftime('%Y-%m-%d %H:%M:%S %Z'))
print('SISTEM :', sys.version.split()[0], platform.platform())
kunci = NIM + '|' + waktu.strftime('%Y-%m-%d')
print('TOKEN  :', hashlib.sha256(kunci.encode()).hexdigest()[:16])


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import math
import os


### E-1 Percobaan Histogram


In [ ]:
# Fungsi buat ngitung histogram manual pake for loop biasa
def hitung_hist_manual(img):
    hist = np.zeros(256, dtype=np.int64)
    tinggi, lebar = img.shape
    for i in range(tinggi):
        for j in range(lebar):
            hist[img[i, j]] += 1
    return hist

def cek_histogram(path, nama_file):
    img_gray = cv.imread(path, cv.IMREAD_GRAYSCALE)
    if img_gray is None:
        print("Yah, gambarnya gak ketemu nih:", path)
        return
    
    # Hitung pake 3 cara
    h_manual = hitung_hist_manual(img_gray)
    h_np, _ = np.histogram(img_gray.ravel(), bins=256, range=(0, 256))
    h_cv = cv.calcHist([img_gray], [0], None, [256], [0, 256]).ravel().astype(np.int64)
    
    # Bandingin buat cari selisih max
    diff_np = np.max(np.abs(h_manual - h_np))
    diff_cv = np.max(np.abs(h_manual - h_cv))
    
    print(f"--- Hasil perbandingan {nama_file} ---")
    print(f"Selisih manual dgn numpy: {diff_np}")
    print(f"Selisih manual dgn cv2  : {diff_cv}\n")

# tes ke lena
cek_histogram('/content/drive/MyDrive/pcvk_2026/lena.jpg', 'lena.jpg')
# tes ke foto ktm (karena belom ada 4 foto, tes pake foto aktivitas jg bisa)
cek_histogram('/content/drive/MyDrive/pcvk_2026/foto_ktm.jpg', 'foto_ktm.jpg')


**Jawaban Pertanyaan E1:**
Hasil perhitungan dari ketiga cara di atas hasilnya 100% sama (selisih maksimumnya 0). Secara logika ketiganya ngelakuin hal yang persis sama, yaitu ngitung jumlah masing-masing warna/intensitas piksel dari 0 sampai 255. Bedanya cuma dari segi kecepatan eksekusinya. Kalau pakai *looping* manual bawaan Python bakal lama banget karena harus lewat piksel satu per satu, beda sama `numpy.histogram` atau `cv.calcHist` yang pemrosesannya dibantu modul bahasa C jadi jauh lebih kenceng (teroptimasi).


In [ ]:
# Plot Histogram dan CDF
list_gambar = [
    ('ktm_gelap.jpg', 'Kondisi Gelap'),
    ('ktm_ruang.jpg', 'Lampu Ruangan'),
    ('ktm_flash.jpg', 'Pake Flash'),
    ('ktm_terang.jpg', 'Cahaya Matahari')
]

plt.figure(figsize=(14, 10))
plt.suptitle('Grafik p(j) dan CDF buat 4 kondisi cahaya')

for idx, (nama, label) in enumerate(list_gambar):
    path = f'/content/drive/MyDrive/pcvk_2026/{nama}'
    img = cv.imread(path, cv.IMREAD_GRAYSCALE)
    
    plt.subplot(2, 2, idx+1)
    if img is not None:
        h = hitung_hist_manual(img)
        pj = h / (img.shape[0] * img.shape[1]) # normalisasi
        cdf = np.cumsum(pj)
        
        plt.plot(pj, color='blue', label='p(j)')
        plt.plot(cdf, color='red', label='CDF')
        plt.title(f"{nama} - {label} (rata2 kecerahan: {img.mean():.1f})")
        plt.legend()
    else:
        plt.title(f"File {nama} belum diupload ke Drive")
        plt.axis('off')

plt.tight_layout()
plt.show()


**Analisa Bentuk Kurva CDF vs Pencahayaan:**
- Untuk foto yang **paling gelap**, sebagian besar nilai intensitas numpuk di nilai-nilai kecil (sebelah kiri). Makanya kurva CDF-nya langsung melesat naik di awal dan cepet banget nyentuh angka 1. Ini nandain informasinya kurang akibat minim cahaya.
- Buat foto yang **paling terang**, nilai intensitasnya malah numpuk di sebelah kanan (deket angka 255 putih). Efeknya, grafik CDF di bagian awal bakal datar aja, terus baru naik drastis pas di akhir grafik. Ini wajar karena fotonya terlalu terang (overexposure).
- Nah, kurva CDF itu mencerminkan sebaran cahayanya. Kalau grafiknya nanjak pelan-pelan (landai) dari ujung ke ujung (seperti foto di lampu ruangan), berarti penyebaran cahayanya seimbang dan pas.


In [ ]:
img_ktm = cv.imread('/content/drive/MyDrive/pcvk_2026/foto_ktm.jpg', 0)

if img_ktm is not None:
    plt.figure(figsize=(14, 5))
    
    # plot dgn jumlah bins berdasarkan perhitungan rumus Hafis (bins = 16)
    plt.subplot(1, 2, 1)
    plt.hist(img_ktm.ravel(), bins=PARAM['bins'], range=(0, 256), color='green', edgecolor='black', alpha=0.8)
    plt.title(f"Histogram pake {PARAM['bins']} bins")
    plt.xlabel('Intensitas')
    
    # plot pake 256 bins full
    plt.subplot(1, 2, 2)
    plt.hist(img_ktm.ravel(), bins=256, range=(0, 256), color='purple', alpha=0.8)
    plt.title("Histogram pake 256 bins full")
    plt.xlabel('Intensitas')
    
    plt.show()


**Informasi yang hilang saat bins dikurangi:**
Karena nilai bins saya = 16, artinya rentang nilai dari 0-255 dibagi jadi 16 kotak aja (masing-masing kotak nampung sekitar 16 angka intensitas). Akibatnya, kita nggak bisa lagi ngeliat fluktuasi detail perbedaan warna piksel per satuan nilainya. Semacam informasinya jadi "dipukul rata", sehingga lekukan kecil atau puncak tajam spesifik di grafiknya jadi ketutupan. Intinya gambaran besar warnanya masih dapet, tapi detail-detail halusnya hilang.


### E-2 Histogram Equalization (HE)


In [ ]:
def he_manual_hafis(img):
    # proses HE step by step manual
    h = hitung_hist_manual(img)
    pj = h / img.size
    cdf = np.cumsum(pj)
    # rumus mapping
    mapping_warna = np.round(cdf * 255).astype(np.uint8)
    # terapkan mapping ke gambar aslinya
    return mapping_warna[img]

def uji_coba_he(path, nama):
    img = cv.imread(path, 0)
    if img is None: return
    
    img_he_manual = he_manual_hafis(img)
    img_he_cv = cv.equalizeHist(img)
    
    selisih = np.max(np.abs(img_he_manual.astype(int) - img_he_cv.astype(int)))
    print(f"Selisih hasil HE manual vs cv.equalizeHist buat {nama} = {selisih}")
    
    plt.figure(figsize=(10,4))
    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap='gray')
    plt.title('Citra Original')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(img_he_manual, cmap='gray')
    plt.title('Hasil HE Manual')
    plt.axis('off')
    plt.show()

uji_coba_he('/content/drive/MyDrive/pcvk_2026/lena_lc.jpg', 'lena_lc')
# Kalo punya ktm yg gelap bisa dites di sini
uji_coba_he('/content/drive/MyDrive/pcvk_2026/ktm_gelap.jpg', 'ktm_gelap') 


In [ ]:
# HE di Citra Berwarna per Kanal B,G,R
img_color = cv.imread('/content/drive/MyDrive/pcvk_2026/lena.jpg')
if img_color is not None:
    b, g, r = cv.split(img_color)
    
    # Ekualisasi satu persatu warnanya
    b_he = cv.equalizeHist(b)
    g_he = cv.equalizeHist(g)
    r_he = cv.equalizeHist(r)
    
    img_bgr_he = cv.merge((b_he, g_he, r_he))
    
    plt.figure(figsize=(12,5))
    plt.subplot(1, 2, 1)
    plt.imshow(cv.cvtColor(img_color, cv.COLOR_BGR2RGB))
    plt.title('Original (Kanal BGR)')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(cv.cvtColor(img_bgr_he, cv.COLOR_BGR2RGB))
    plt.title('HE Terpisah tiap B, G, R')
    plt.axis('off')
    plt.show()


In [ ]:
def itung_geser_hue(ori, manipulasi):
    hue_ori = cv.cvtColor(ori, cv.COLOR_BGR2HSV)[:,:,0].astype(int)
    hue_baru = cv.cvtColor(manipulasi, cv.COLOR_BGR2HSV)[:,:,0].astype(int)
    jarak = np.abs(hue_ori - hue_baru)
    jarak = np.minimum(jarak, 180 - jarak) # efek melingkar derajat
    return jarak.mean() * 2

# Cobain ke KTM terang atau gambar lain
img_test = cv.imread('/content/drive/MyDrive/pcvk_2026/foto_ktm.jpg')
if img_test is None:
    img_test = cv.imread('/content/drive/MyDrive/pcvk_2026/lena.jpg')

if img_test is not None:
    # 1. YCrCb
    y, cr, cb = cv.split(cv.cvtColor(img_test, cv.COLOR_BGR2YCrCb))
    he_ycrcb = cv.cvtColor(cv.merge([cv.equalizeHist(y), cr, cb]), cv.COLOR_YCrCb2BGR)
    
    # 2. HSV
    h, s, v = cv.split(cv.cvtColor(img_test, cv.COLOR_BGR2HSV))
    he_hsv = cv.cvtColor(cv.merge([h, s, cv.equalizeHist(v)]), cv.COLOR_HSV2BGR)
    
    # 3. LAB
    l, a, b = cv.split(cv.cvtColor(img_test, cv.COLOR_BGR2LAB))
    he_lab = cv.cvtColor(cv.merge([cv.equalizeHist(l), a, b]), cv.COLOR_LAB2BGR)
    
    # 4. BGR dipisah kek tadi
    b_t, g_t, r_t = cv.split(img_test)
    he_bgr_2 = cv.merge([cv.equalizeHist(b_t), cv.equalizeHist(g_t), cv.equalizeHist(r_t)])
    
    citra_list = [img_test, he_bgr_2, he_ycrcb, he_hsv, he_lab]
    nama_list = ['Asli', 'HE (BGR)', 'HE (Kanal Y)', 'HE (Kanal V)', 'HE (Kanal L)']
    
    plt.figure(figsize=(20, 4))
    for i in range(5):
        plt.subplot(1, 5, i+1)
        plt.imshow(cv.cvtColor(citra_list[i], cv.COLOR_BGR2RGB))
        plt.title(nama_list[i])
        plt.axis('off')
    plt.show()
    
    print(f"{'Metode':<15} | {'Geser Hue (derajat)':<20} | {'Rata2 Terang'}")
    print("-" * 55)
    for nm, gbr_res in zip(nama_list, citra_list):
        geser = itung_geser_hue(img_test, gbr_res)
        terang = cv.cvtColor(gbr_res, cv.COLOR_BGR2GRAY).mean()
        print(f"{nm:<15} | {geser:<20.2f} | {terang:.1f}")


**Jawaban tentang Pergeseran Hue dan Kanal Terbaik:**
Kelihatan banget dari tabel di atas kalau metode **HE di tiap B, G, R terpisah** itu menghasilkan pergeseran Hue yang paling nyeleneh (besar banget nilainya). Alasannya simpel: karena warna B, G, dan R diperlakukan masing-masing tanpa ngeliat warna lain, rasio perpaduan warna aslinya jadi hancur. 

Terus kenapa pas kita cuma HE di kanal terang (kayak Y/V/L) pergeseran Huenya gak murni 0? Itu efek dari komputasinya opencv aja, waktu dikonversi ke sana kemari ada pembulatan angka (rounding errors), jadi pas dibalikin ke BGR warnanya sedikit meleset angkanya.

Menurut saya kanal **L (dari LAB)** itu yang paling mantap dan pas buat foto KTM karena ruang warna LAB ini didesain paling sesuai sama penglihatan manusia. Jadi meskipun kontras kecerahannya ditingkatin, warna aslinya keliatan tetep terjaga secara visual.


In [ ]:
# Coba CLAHE di kanal L (menggunakan parameter N = 5 -> clip = 1.0, tile = 3)
if img_test is not None:
    lab_color = cv.cvtColor(img_test, cv.COLOR_BGR2LAB)
    l_ch, a_ch, b_ch = cv.split(lab_color)
    
    # bikin objek clahe
    clahe_hafis = cv.createCLAHE(clipLimit=PARAM['clip'], tileGridSize=(PARAM['tile'], PARAM['tile']))
    l_clahe_hasil = clahe_hafis.apply(l_ch)
    
    img_clahe_akhir = cv.cvtColor(cv.merge([l_clahe_hasil, a_ch, b_ch]), cv.COLOR_LAB2BGR)
    
    shift_clahe = itung_geser_hue(img_test, img_clahe_akhir)
    bright_clahe = cv.cvtColor(img_clahe_akhir, cv.COLOR_BGR2GRAY).mean()
    
    print(f"Hasil HE Biasa (Kanal L) -> Geser Hue: {itung_geser_hue(img_test, he_lab):.2f}, Terang: {cv.cvtColor(he_lab, cv.COLOR_BGR2GRAY).mean():.1f}")
    print(f"Hasil CLAHE (Kanal L)    -> Geser Hue: {shift_clahe:.2f}, Terang: {bright_clahe:.1f}")
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(cv.cvtColor(he_lab, cv.COLOR_BGR2RGB))
    plt.title('HE Biasa (L)')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(cv.cvtColor(img_clahe_akhir, cv.COLOR_BGR2RGB))
    plt.title(f"CLAHE (clip={PARAM['clip']}, tile={PARAM['tile']})")
    plt.axis('off')
    plt.show()


In [ ]:
# Hitung MSE sama PSNR
img_buat_psnr = cv.imread('/content/drive/MyDrive/pcvk_2026/ktm_gelap.jpg', 0)
if img_buat_psnr is None:
    img_buat_psnr = cv.imread('/content/drive/MyDrive/pcvk_2026/lena_lc.jpg', 0)

if img_buat_psnr is not None:
    hasil_he_global = cv.equalizeHist(img_buat_psnr)
    
    # mse manual
    mse = np.mean((img_buat_psnr.astype(float) - hasil_he_global.astype(float)) ** 2)
    psnr = 10 * math.log10((255**2) / mse) if mse > 0 else 0
        
    print(f"MSE-nya  : {mse:.3f}")
    print(f"PSNR-nya : {psnr:.3f} dB")


**Kenapa PSNR-nya jelek? (Jawaban 1c)**
Nilai PSNR bisa rendah banget tuh dikarenakan selisih piksel (MSE) yang membengkak. Proses HE 'kan kerjaannya memang ngerubah nilai gelap jadi terang secara ekstrem biar gambarnya lebih kelihatan. Konsekuensinya, selisih secara matematis sama aslinya ya beda jauh. Kesimpulannya, jangan ngandelin PSNR murni buat ngukur tingkat kualitas foto hasil pencerahan kayak HE ini, karena seringkali secara visual jauh lebih gampang dibaca mata kita, tapi di mata rumus PSNR itu dianggap "penyimpangan" / error.


In [ ]:
if img_buat_psnr is not None:
    # CLAHE pake parameter Hafis
    clahe_default = cv.createCLAHE(clipLimit=PARAM['clip'], tileGridSize=(PARAM['tile'], PARAM['tile']))
    hasil_clahe_dft = clahe_default.apply(img_buat_psnr)
    
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1); plt.imshow(img_buat_psnr, cmap='gray'); plt.title('Asli'); plt.axis('off')
    plt.subplot(1, 3, 2); plt.imshow(hasil_he_global, cmap='gray'); plt.title('HE Global'); plt.axis('off')
    plt.subplot(1, 3, 3); plt.imshow(hasil_clahe_dft, cmap='gray'); plt.title(f"CLAHE (clip={PARAM['clip']})"); plt.axis('off')
    plt.show()

    # Testing beberapa clipLimit di sekitar nilai clip hafis (1.0)
    clip_tests = [0.5, PARAM['clip'], 2.5, 4.0]
    plt.figure(figsize=(16, 5))
    for i, batas_clip in enumerate(clip_tests):
        cl = cv.createCLAHE(clipLimit=batas_clip, tileGridSize=(PARAM['tile'], PARAM['tile']))
        hasil_batas = cl.apply(img_buat_psnr)
        
        plt.subplot(1, len(clip_tests), i+1)
        plt.imshow(hasil_batas, cmap='gray')
        plt.title(f"clipLimit = {batas_clip}")
        plt.axis('off')
    plt.show()


**Jawaban Praktikum CLAHE (2a & 2b):**
- **2a**: Kenapa CLAHE bikin tulisan di foto identitas jadi lebih jelas dibaca? Karena CLAHE itu nggak narik kontras se-gambar penuh (global), melainkan dibagi-bagi per "tile" atau zona kecil-kecil. Teks nama kita jadi kebaca karena di area situ aja kontrasnya dipoles.
- **2b**: Kalo nilai parameter `clipLimit` dinaikkin terus, kontrasnya bakal terangkat pol, yang berakibat kotoran atau *noise* di gambar juga ikut ke-boost banget, gambar jadi kelihatan kasar/bercak-bercak. Nilai yang pas kayak `clipLimit = 1.0` (angka dari rumus NIM saya) rasanya pas banget; tulisannya ngebentuk jelas tanpa narik *noise* lebay.


### E-3 Dithering pake Floyd-Steinberg


In [ ]:
def floyd_steinberg_hafis(citra, jum_L):
    img = citra.copy().astype(float)
    step = 255.0 / (jum_L - 1)
    H, W = img.shape
    
    for y in range(H):
        for x in range(W):
            nilai_lama = img[y, x]
            # buletin ke step terdekat
            nilai_baru = np.round(nilai_lama / step) * step
            img[y, x] = nilai_baru
            
            selisih_error = nilai_lama - nilai_baru
            
            # lempar errornya ke tetangga biar gradasinya mulus
            if x + 1 < W:
                img[y, x+1] += selisih_error * 7/16
            if y + 1 < H:
                if x - 1 >= 0:
                    img[y+1, x-1] += selisih_error * 3/16
                img[y+1, x] += selisih_error * 5/16
                if x + 1 < W:
                    img[y+1, x+1] += selisih_error * 1/16
                    
    return np.clip(img, 0, 255).astype(np.uint8)

# Tes lena dulu L=2
img_lena = cv.imread('/content/drive/MyDrive/pcvk_2026/lena.jpg', 0)
if img_lena is not None:
    lena_dither = floyd_steinberg_hafis(img_lena, 2)
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1); plt.imshow(img_lena, cmap='gray'); plt.title('Lena Asli'); plt.axis('off')
    plt.subplot(1, 2, 2); plt.imshow(lena_dither, cmap='gray'); plt.title('Dithering L=2'); plt.axis('off')
    plt.show()

# Tes di KTM pake rumus hafiz (L = 8)
img_ktm_tes = cv.imread('/content/drive/MyDrive/pcvk_2026/foto_ktm.jpg', 0)
if img_ktm_tes is not None:
    ktm_dither = floyd_steinberg_hafis(img_ktm_tes, PARAM['L'])
    
    # Cobain kombinasi: HE trus Dithering
    ktm_dihe = cv.equalizeHist(img_ktm_tes)
    ktm_he_dither = floyd_steinberg_hafis(ktm_dihe, PARAM['L'])
    
    plt.figure(figsize=(18, 5))
    plt.subplot(1, 4, 1); plt.imshow(img_ktm_tes, cmap='gray'); plt.title('1. Foto Asli'); plt.axis('off')
    plt.subplot(1, 4, 2); plt.imshow(ktm_dither, cmap='gray'); plt.title(f"2. Langsung Dithering (L={PARAM['L']})"); plt.axis('off')
    plt.subplot(1, 4, 3); plt.imshow(ktm_dihe, cmap='gray'); plt.title('3. Di HE Global Dulu'); plt.axis('off')
    plt.subplot(1, 4, 4); plt.imshow(ktm_he_dither, cmap='gray'); plt.title('4. Habis HE -> Dithering'); plt.axis('off')
    plt.show()


**Kesimpulan Urutan HE trus Dithering:**
Kalo kita punya foto yang kontrasnya jelek (apalagi gelap), kalo dilangsungin *dithering*, titik-titik dither-nya bakal numpuk jadi gelap semua dan alhasil tulisannya nyaru (gak kebaca). Tapi, kalo fotonya di-*Histogram Equalization* dulu, kontras rentang cahayanya otomatis melar/terbagi rata. Nah pas dikenain dithering setelahnya, sebaran titik-titik hitam putihnya dapet bahan intensitas yang lebih bervariasi. Jadinya teks atau ukiran di foto KTM kita polanya bakal jauh lebih jelas dan enak dibaca.
